In [1]:
!pip install ecmwf-api-client

  Using cached ecmwf_api_client-1.6.5-py3-none-any.whl.metadata (4.8 kB)
Using cached ecmwf_api_client-1.6.5-py3-none-any.whl (13 kB)


In [2]:
#!/usr/bin/env python
from ecmwfapi import ECMWFDataServer

server = ECMWFDataServer()

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2016-02-01/to/2016-02-29",
    "expver": "prod",
    "grid": "0.5/0.5",
    "levtype": "sfc",
    "origin": "ecmf",
    "param": "228228",
    "step": "0/24/48/72/96/120/144/168/192/216/240/264/288/312/336/360",
    "time": "00:00:00/12:00:00",
    "type": "cf",
    "target": "data",
    "area": "30/90/-15/140"
})

2025-11-15 15:02:51 ECMWF API python library 1.6.5
2025-11-15 15:02:51 ECMWF API at https://api.ecmwf.int/v1
2025-11-15 15:02:55 Welcome Mochammad Zharif
2025-11-15 15:02:58 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2025-11-15 15:02:58 ------------ WARNING ------------
2025-11-15 15:02:58 Access to this dataset is transitioning to a new interface, dates to be announced soon
2025-11-15 15:02:58 For more information on how to access this data in the future, visit https://confluence.ecmwf.int/x/-wUiEw
2025-11-15 15:02:58 ---------------------------------
2025-11-15 15:02:59 Request submitted
2025-11-15 15:02:59 Request id: 691833b36513bd99e462eccb
2025-11-15 15:02:59 Request is submitted
2025-11-15 15:03:02 Calling 'nice mars /tmp/20251115-0800/13/tmp-_mars-W0JxFD-db605fb525e773dcee291b28b61a2307.req'
2025-11-15 15:03:02 Forcing MIR_CACHE_PATH=/data/ec_coeff
2025-11-15 15:03:02 mars - WARN -
2025-11-15 15:03:

In [ ]:
#!/usr/bin/env python
from ecmwfapi import ECMWFDataServer
from datetime import datetime, timedelta
import os

server = ECMWFDataServer()

# Output folder
output_dir = "../../../data/ifs-1.5deg-tigge"
os.makedirs(output_dir, exist_ok=True)

year = 2017

for month in range(2, 13):

    # Start of month
    start_date = datetime(year, month, 1)

    # End of month
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)

    # Format dates
    start_str = start_date.strftime("%Y-%m-%d")
    end_str   = end_date.strftime("%Y-%m-%d")

    date_range = f"{start_str}/to/{end_str}"

    # Output filename per month
    target_file = f"{output_dir}/{year}-{month:02d}.grib"

    print(f"Downloading {date_range} → {target_file}")

    try:
        server.retrieve({
            "class": "ti",
            "dataset": "tigge",
            "date": date_range,
            "expver": "prod",
            "grid": "1.5/1.5",
            "levtype": "sfc",
            "origin": "ecmf",
            "param": "165/166/167/228228",
            "step": "0/24/48/72/96/120/144/168/192/216/240/264/288/312/336/360",
            "time": "00:00:00/12:00:00",
            "type": "cf",
            "target": target_file,
            "area": "30/90/-15/140"
        })
    except Exception as e:
        print(f"Error downloading {date_range}: {e}")

print("Done.")


2026-01-13 16:36:35 ECMWF API python library 1.6.5
2026-01-13 16:36:35 ECMWF API at https://api.ecmwf.int/v1
2026-01-13 16:36:35 Welcome Mochammad Zharif
2026-01-13 16:36:37 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2026-01-13 16:36:37 ------------ WARNING ------------
2026-01-13 16:36:37 Access to this dataset is transitioning to a new interface, dates to be announced soon
2026-01-13 16:36:37 For more information on how to access this data in the future, visit https://confluence.ecmwf.int/x/-wUiEw
2026-01-13 16:36:37 ---------------------------------
2026-01-13 16:36:38 Request submitted
2026-01-13 16:36:38 Request id: 69661226783efb178d1cccca
2026-01-13 16:36:38 Request is submitted
2026-01-13 16:36:40 Calling 'nice mars /tmp/20260113-0930/ef/tmp-_mars-vPJQKM-06fd331b451f70b64f41f0ebc030446f.req'
2026-01-13 16:36:40 Forcing MIR_CACHE_PATH=/data/ec_coeff
2026-01-13 16:36:40 mars - WARN -
2026-01-13 16:36:

In [ ]:
#!/usr/bin/env python
from ecmwfapi import ECMWFDataServer
from datetime import datetime, timedelta
from tqdm import tqdm
import os

server = ECMWFDataServer()

# --- CONFIGURATION ---
output_dir = "../../../data/s2s"
os.makedirs(output_dir, exist_ok=True)

# Use 2025 to ensure we are requesting past data that definitely exists
target_year = 2025
start_date = datetime(target_year, 1, 1)
end_date = datetime(target_year, 12, 31)

# --- STEP GENERATION ---
steps_maxtemp = "/".join([str(i) for i in range(6, 1105, 6)])
steps_standard = "0/" + steps_maxtemp

# --- HELPER FUNCTION ---
def get_hdate_fixed_range(current_date, start_year=2016, end_year=2025):
    hdates = []
    for year in range(end_year, start_year - 1, -1):
        try:
            h_date = current_date.replace(year=year)
            hdates.append(h_date.strftime("%Y-%m-%d"))
        except ValueError:
            h_date = current_date.replace(year=year, day=28)
            hdates.append(h_date.strftime("%Y-%m-%d"))
    return "/".join(hdates)

# --- MAIN LOOP ---
current = start_date
total_days = (end_date - start_date).days + 1

with tqdm(total=total_days, desc="Checking Dates", unit="day") as pbar:
    
    while current <= end_date:
        
        if current.weekday() in [0, 3]: # Mondays and Thursdays
            
            date_str = current.strftime("%Y-%m-%d")
            hdate_str = get_hdate_fixed_range(current, start_year=2016, end_year=2025)
            target_file_base = f"{output_dir}/s2s_{date_str}"
            
            pbar.write(f"\n[+] Processing Model Date: {date_str}")
            
            try:
                # REQUEST 1: Max Temp
                # ADDED: "expect": "any"
                server.retrieve({
                    "class": "s2",
                    "dataset": "s2s",
                    "date": date_str,
                    "expver": "prod",
                    "hdate": hdate_str,
                    "levtype": "sfc",
                    "model": "glob",
                    "origin": "ecmf",
                    "stream": "enfh",
                    "type": "cf",
                    "area": "30/90/-15/140",
                    "param": "121",
                    "step": steps_maxtemp,
                    "time": "00:00:00",
                    "expect": "any", # <--- CRITICAL FIX
                    "target": f"{target_file_base}_maxt.grib"
                })

                # REQUEST 2: Wind & Rain
                # ADDED: "expect": "any"
                server.retrieve({
                    "class": "s2",
                    "dataset": "s2s",
                    "date": date_str,
                    "expver": "prod",
                    "hdate": hdate_str,
                    "levtype": "sfc",
                    "model": "glob",
                    "origin": "ecmf",
                    "stream": "enfh",
                    "type": "cf",
                    "area": "30/90/-15/140",
                    "param": "165/166/228",
                    "step": steps_standard,
                    "time": "00:00:00",
                    "expect": "any", # <--- CRITICAL FIX
                    "target": f"{target_file_base}_surf.grib"
                })
                
                pbar.write(f"    [OK] Success: {date_str}")

            except Exception as e:
                pbar.write(f"    [!] Error downloading {date_str}: {e}")

        current += timedelta(days=1)
        pbar.update(1)

print("Done.")

In [2]:
#!/usr/bin/env python
from ecmwfapi import ECMWFDataServer
from datetime import datetime, timedelta
import os

server = ECMWFDataServer()

# Output folder
output_dir = "../../../data/ifs-1.5deg-tigge"
os.makedirs(output_dir, exist_ok=True)

def download_data(date_range, target_file, max_retries=2):
    """Download data with retry mechanism"""
    for attempt in range(max_retries):
        try:
            server.retrieve({
                "class": "ti",
                "dataset": "tigge",
                "date": date_range,
                "expver": "prod",
                "grid": "1.5/1.5",
                "levtype": "sfc",
                "origin": "ecmf",
                "param": "165/166/167/228228",
                "step": "0/24/48/72/96/120/144/168/192/216/240/264/288/312/336/360",
                "time": "00:00:00/12:00:00",
                "type": "cf",
                "target": target_file,
                "area": "30/90/-15/140"
            })
            return True
        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                print("  Retrying...")
            else:
                print("  All retries failed")
    return False

def download_daily(year, month, output_dir):
    """Download data day by day for a given month"""
    print(f"  Falling back to daily downloads for {year}-{month:02d}")
    
    # Get the number of days in the month
    if month == 12:
        next_month = datetime(year + 1, 1, 1)
    else:
        next_month = datetime(year, month + 1, 1)
    
    days_in_month = (next_month - datetime(year, month, 1)).days
    
    daily_files = []
    failed_days = []
    
    for day in range(1, days_in_month + 1):
        date = datetime(year, month, day)
        date_str = date.strftime("%Y-%m-%d")
        daily_file = f"{output_dir}/{year}-{month:02d}-{day:02d}.grib"
        
        print(f"  Downloading {date_str} → {daily_file}")
        
        if download_data(date_str, daily_file, max_retries=2):
            daily_files.append(daily_file)
        else:
            print(f"  Failed to download {date_str}")
            failed_days.append(date_str)
    
    return daily_files, failed_days

def merge_daily_files(daily_files, target_file):
    """Merge daily GRIB files into a single monthly file"""
    if not daily_files:
        return False
    
    try:
        print(f"  Merging {len(daily_files)} daily files into {target_file}")
        # Simple concatenation for GRIB files
        with open(target_file, 'wb') as outfile:
            for daily_file in daily_files:
                with open(daily_file, 'rb') as infile:
                    outfile.write(infile.read())
                # Optionally remove daily file after merging
                os.remove(daily_file)
        print(f"  Successfully merged into {target_file}")
        return True
    except Exception as e:
        print(f"  Error merging files: {e}")
        return False

year = 2017
for month in range(1, 13):
    # Start of month
    start_date = datetime(year, month, 1)
    
    # End of month
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)
    
    # Format dates
    start_str = start_date.strftime("%Y-%m-%d")
    end_str   = end_date.strftime("%Y-%m-%d")
    date_range = f"{start_str}/to/{end_str}"
    
    # Output filename per month
    target_file = f"{output_dir}/{year}-{month:02d}.grib"
    
    print(f"\nDownloading {date_range} → {target_file}")
    
    # Try monthly download first
    if download_data(date_range, target_file):
        print(f"✓ Successfully downloaded {year}-{month:02d}")
    else:
        # Fall back to daily downloads
        print(f"✗ Monthly download failed for {year}-{month:02d}")
        daily_files, failed_days = download_daily(year, month, output_dir)
        
        if daily_files:
            # Merge daily files into monthly file
            if merge_daily_files(daily_files, target_file):
                if failed_days:
                    print(f"⚠ Warning: Missing data for days: {', '.join(failed_days)}")
                else:
                    print(f"✓ Successfully downloaded all days for {year}-{month:02d}")
            else:
                print(f"✗ Failed to merge daily files for {year}-{month:02d}")
        else:
            print(f"✗ No daily data could be downloaded for {year}-{month:02d}")

print("\nDone.")


2026-01-18 08:18:01 ECMWF API python library 1.6.5
2026-01-18 08:18:01 ECMWF API at https://api.ecmwf.int/v1
2026-01-18 08:18:02 Welcome Mochammad Zharif
2026-01-18 08:18:04 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2026-01-18 08:18:04 ------------ WARNING ------------
2026-01-18 08:18:04 Access to this dataset is transitioning to a new interface, dates to be announced soon
2026-01-18 08:18:04 For more information on how to access this data in the future, visit https://confluence.ecmwf.int/x/-wUiEw
2026-01-18 08:18:04 ---------------------------------
2026-01-18 08:18:04 Request submitted
2026-01-18 08:18:04 Request id: 696c34cc4100f2f8c4630412
2026-01-18 08:18:04 Request is submitted
2026-01-18 08:18:06 Calling 'nice mars /tmp/20260118-0110/b1/tmp-_mars-Z1C3Yx-96395b1319cfed28534ca18b4064b9fd.req'
2026-01-18 08:18:06 Forcing MIR_CACHE_PATH=/data/ec_coeff
2026-01-18 08:18:06 mars - WARN -
2026-01-18 08:18

KeyboardInterrupt: 